# OmicsDRP Inference Model Walkthrough

이 노트북은 **학습을 다시 돌리지 않고**, 이미 학습된 OmicsDRP inference ensemble
모델(가중치)만 불러와서 downstream 분석(임베딩 분석, 간단한 설명가능성 분석 등)을
시작하기 위한 예시입니다.

## 필요한 것 (전달받아야 할 파일)

1. **코드**: `omicsdrp/src/omicsdrp/` 패키지 전체 (모델 구조 정의)
2. **가중치**: `omicsdrp/scripts/InferenceModels/<condition_tag>/fold_k.pt` (5개 fold)
   - 이 폴더 하나가 "조건 하나"(사용한 omics 조합 + cell encoder + drug encoder + split
     방식)에 대한 앙상블입니다. `config.json`도 같이 있어야 합니다.
3. **데이터**: `data/` 폴더 (최소 `PGKB_Gene_data_dict.pth`, `IC50_GDSC2.csv`,
   `TargetDrugs_with_MorganFingerprint_GDSC2_512.txt`, `gene_list.txt`,
   `Cell_line_meta.csv`). pretrained 드러그 인코더(chemberta/molformer/graphormer/unimol)
   조건을 쓸 경우 `data/drug_embeddings/`도 필요합니다.
4. **환경**: `conda activate omicsdrp` (torch, pandas, sklearn, rdkit 등). 이 노트북
   추가로 `scikit-learn`(PCA)과 선택적으로 `umap-learn`을 사용합니다.

## 중요: 이 가중치가 "성능 숫자"는 아님

`InferenceModels`의 fold 점수는 **early-stopping에 쓰인 fold**에서 나온 것이라
낙관적으로 편향돼 있습니다 (`CLAUDE.md` 참고). 논문/리포트용 성능 숫자는 nested-CV
결과(`omicsdrp/scripts/Results`)를 봐야 하고, 이 가중치들은 **새로운 데이터에 대한
추론**과 이 노트북 같은 **탐색적 downstream 분석**에만 쓰는 용도입니다.


In [ ]:
import sys, os, glob, json

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
SRC = os.path.join(REPO_ROOT, "omicsdrp", "src")
DATASET_PATH = os.path.join(REPO_ROOT, "data")
INFERENCE_ROOT = os.path.join(REPO_ROOT, "omicsdrp", "scripts", "InferenceModels")

sys.path.insert(0, SRC)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from omicsdrp.data import load_raw, select_omics, stack_gene_data
from omicsdrp.inference_models import InferenceEnsemble, _apply_saved_scaler

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
print("dataset path:", DATASET_PATH)
print("inference models root:", INFERENCE_ROOT)


## 1. 사용 가능한 inference 조건 목록 확인

`InferenceModels/` 아래 폴더 하나 = 조건 하나(`omics__cell_encoder__drug_encoder__split_mode__hash`).
가장 해석하기 쉬운 예시로 **전체 omics(SNP+MET+CNV+RNA) + attention cell encoder + morgan drug encoder** 조건을 기본값으로 고릅니다 (attention 조건이어야 아래 8절의
attention-map 분석이 가능합니다). 다른 조건을 보고 싶으면 `CONDITION_TAG`를 바꾸세요.

In [ ]:
available = sorted(
    d for d in os.listdir(INFERENCE_ROOT)
    if os.path.isdir(os.path.join(INFERENCE_ROOT, d))
)
print(f"{len(available)} conditions found:")
for c in available:
    print(" -", c)

# 기본값: 전체 omics + attention + morgan 조건을 우선 사용, 없으면 첫 번째 조건 사용
preferred = [c for c in available if c.startswith("SNP+MET+CNV+RNA__attention__morgan")]
CONDITION_TAG = preferred[0] if preferred else available[0]
COND_DIR = os.path.join(INFERENCE_ROOT, CONDITION_TAG)
print("\nusing condition:", CONDITION_TAG)

with open(os.path.join(COND_DIR, "config.json")) as fh:
    print(json.dumps(json.load(fh), indent=2))


## 2. 모델 구조 + 가중치 불러오기

`InferenceEnsemble.load()`가 `config.json`(모델 구조 정의)과 `fold_k.pt`(각 fold의
가중치 + 그 fold가 학습 때 쓴 gene scaler)를 함께 불러옵니다. 5개 fold 각각이
독립적으로 학습된 모델이고, `predict()`는 5개의 예측을 평균 낸 앙상블 결과를 줍니다.

In [ ]:
ensemble = InferenceEnsemble.load(COND_DIR, dataset_path=DATASET_PATH, device=DEVICE)
print(f"loaded {len(ensemble.folds)} fold models for condition '{CONDITION_TAG}'")
print("omics used:", ensemble.config.omics, "-> indices", ensemble.config.omics_indices())
print("cell_encoder:", ensemble.config.cell_encoder, "| drug_encoder:", ensemble.config.drug_encoder)


## 3. 원본 데이터 불러오기

`load_raw`는 세포주별 per-gene omics 텐서, 약물 메타데이터, IC50 행렬을 불러오고
GDSC2의 중복 약물(SMILES 동일)을 데이터 단에서 병합합니다 (241 -> 231개 고유 약물).
모델이 학습된 것과 **같은 feature space**(같은 909 유전자, 같은 231개 약물 순서)여야
가중치를 그대로 재사용할 수 있습니다 -- 새로운 외부 데이터를 쓰더라도 유전자/약물
정렬을 이 구조에 맞춰야 합니다.

In [ ]:
raw = load_raw(DATASET_PATH)
print("n_cell:", raw.n_cell, "| n_drug:", raw.n_drug, "| n_gene:", len(raw.genes))
print("n_pairs (non-NaN IC50 entries):", len(raw.pairs))
raw.drug_meta.head()


## 4. 앙상블 예측 실행 (모든 관측된 cell-drug 쌍)

여기서는 데이터 전체에 대해 예측을 돌려봅니다. `predict()`는 각 fold가 저장해둔
**자기 fold 전용 scaler**를 새 데이터에 적용한 뒤 5개 모델의 예측을 평균 냅니다.
주의: 이 pairs는 모델이 학습에 썼던 것과 겹치는 관측치일 수 있으므로, 아래 metric은
성능 검증이 아니라 "모델이 잘 로드됐는지" 확인하는 sanity check 용도입니다.

In [ ]:
result = ensemble.predict(raw.gene_data, raw.drug_meta, raw.pairs, ic50=raw.ic50)
print("metrics (sanity check, NOT a held-out performance number):")
for k, v in result["metrics"].items():
    print(f"  {k}: {v:.4f}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(result["true"], result["pred"], s=4, alpha=0.2)
lims = [min(result["true"].min(), result["pred"].min()),
        max(result["true"].max(), result["pred"].max())]
ax.plot(lims, lims, "r--", linewidth=1)
ax.set_xlabel("true LN(IC50)")
ax.set_ylabel("ensemble predicted LN(IC50)")
ax.set_title(f"{CONDITION_TAG}\n(sanity check, not held-out performance)")
plt.tight_layout()
plt.show()


## 5. 세포주(cell) 임베딩 분석

`fold_1`의 cell encoder만 떼어내서 모든 세포주를 인코딩합니다. 각 fold는 자기만의
scaler로 학습됐으므로, 임베딩을 뽑을 때도 그 fold의 scaler를 그대로 써야 fold별
임베딩이 서로 일관됩니다 -- 여기서는 fold 1 하나만 예시로 사용합니다(앙상블 평균은
예측에서만 의미가 있고, 임베딩 공간 평균은 아님).

In [ ]:
from omicsdrp.models import DRPModel

fold_ck = ensemble.folds[0]  # fold_1
genes = fold_ck["genes"]

scaled_gene_data = _apply_saved_scaler(
    raw.gene_data, genes, fold_ck["omics_indices"],
    fold_ck["scaler_mean"], fold_ck["scaler_scale"],
)
gene_tensor = stack_gene_data(scaled_gene_data, genes)  # [N_cell, n_gene, n_omics]

fold1_model = DRPModel(genes, raw.drug_meta, ensemble.config).to(DEVICE)
fold1_model.load_state_dict(fold_ck["model_state"], strict=False)
fold1_model.eval()

with torch.no_grad():
    cell_embeddings = fold1_model.cell_encoder(gene_tensor.to(DEVICE)).cpu().numpy()
print("cell_embeddings shape:", cell_embeddings.shape)  # [N_cell, embedding_dim]


In [ ]:
from sklearn.decomposition import PCA

cell_meta = pd.read_csv(os.path.join(DATASET_PATH, "Cell_line_meta.csv"))
ic50_df = pd.read_csv(os.path.join(DATASET_PATH, "IC50_GDSC2.csv"), index_col=0)
cell_ids = ic50_df.index  # row order matches raw.gene_data / cell_embeddings

cell_tissue = (
    pd.DataFrame({"Model_ID": cell_ids})
    .merge(cell_meta[["Model_ID", "Tissue"]], on="Model_ID", how="left")["Tissue"]
    .fillna("unknown")
)

pca = PCA(n_components=2)
cell_2d = pca.fit_transform(cell_embeddings)

fig, ax = plt.subplots(figsize=(7, 6))
for tissue in cell_tissue.value_counts().head(10).index:
    mask = (cell_tissue == tissue).values
    ax.scatter(cell_2d[mask, 0], cell_2d[mask, 1], s=8, alpha=0.6, label=tissue)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("Cell-line embeddings (fold 1), colored by tissue (top 10)")
ax.legend(fontsize=7, markerscale=2, loc="best")
plt.tight_layout()
plt.show()


## 6. 약물(drug) 임베딩 분석

drug encoder는 세포주와 무관하게 약물 인덱스만으로 임베딩을 만듭니다. `TARGET_PATHWAY`
메타데이터로 색칠해서, 비슷한 작용기전(pathway)의 약물들이 임베딩 공간에서 가까이
모이는지 확인합니다.

In [ ]:
drug_idx_all = torch.arange(len(raw.drug_meta), dtype=torch.long).to(DEVICE)
with torch.no_grad():
    drug_embeddings = fold1_model.drug_encoder(drug_idx_all).cpu().numpy()
print("drug_embeddings shape:", drug_embeddings.shape)  # [N_drug, embedding_dim]

drug_pathway = raw.drug_meta["TARGET_PATHWAY"].fillna("unknown")
drug_2d = PCA(n_components=2).fit_transform(drug_embeddings)

fig, ax = plt.subplots(figsize=(7, 6))
for pathway in drug_pathway.value_counts().head(10).index:
    mask = (drug_pathway == pathway).values
    ax.scatter(drug_2d[mask, 0], drug_2d[mask, 1], s=14, alpha=0.7, label=pathway)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("Drug embeddings (fold 1), colored by TARGET_PATHWAY (top 10)")
ax.legend(fontsize=6, markerscale=1.5, loc="best")
plt.tight_layout()
plt.show()


## 7. 설명가능성(XAI) 1 -- 입력 유전자 중요도 (Gradient x Input saliency)

특정 (세포주, 약물) 쌍 하나를 골라서, 예측값이 어떤 유전자 입력에 가장 민감한지
gradient 기반으로 봅니다. 모델 전체(`fold1_model`)를 통해 미분하므로 cell encoder +
drug encoder + response head를 전부 통과한 최종 예측 기준의 saliency입니다.

In [ ]:
example_sample_idx, example_drug_idx = raw.pairs[0]
print("example cell:", cell_ids[example_sample_idx], "| example drug:",
      raw.drug_meta.iloc[example_drug_idx]["DRUG_NAME"])

x = gene_tensor[example_sample_idx:example_sample_idx + 1].clone().to(DEVICE)
x.requires_grad_(True)
d_idx = torch.tensor([example_drug_idx], dtype=torch.long, device=DEVICE)

pred = fold1_model(x, d_idx)
pred.backward()

# saliency per gene = |gradient| summed over omics channels, for this one example
saliency = x.grad.detach().abs().sum(dim=-1).squeeze(0).cpu().numpy()  # [n_gene]
top_k = 20
top_genes_idx = np.argsort(saliency)[::-1][:top_k]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh([genes[i] for i in top_genes_idx][::-1], saliency[top_genes_idx][::-1])
ax.set_xlabel("|d(prediction)/d(gene input)| (summed over omics channels)")
ax.set_title(f"Top {top_k} salient genes for one (cell, drug) example\n"
             f"predicted LN(IC50) = {pred.item():.3f}")
plt.tight_layout()
plt.show()


## 8. 설명가능성(XAI) 2 -- Attention map (attention cell encoder 전용)

`cell_encoder`가 `attention`인 조건에서는 forward pass 후 `cell_encoder.last_attn`에 유전자-유전자 attention 가중치가 캐시됩니다
(`[batch, n_gene, n_gene]`, multi-head 평균). 특정 유전자가 다른 어떤 유전자들에
주의를 많이 기울이는지 볼 수 있습니다.

In [ ]:
if ensemble.config.cell_encoder != "attention":
    print("이 조건은 attention cell encoder가 아니라서 이 섹션은 건너뜁니다.")
else:
    with torch.no_grad():
        _ = fold1_model.cell_encoder(x.detach())
    attn = fold1_model.cell_encoder.last_attn.squeeze(0).cpu().numpy()  # [n_gene, n_gene]

    # 특정 유전자 하나가 받는 attention(다른 유전자로부터 얼마나 주목받는지) 상위 목록
    focus_gene = genes[top_genes_idx[0]]
    focus_idx = genes.index(focus_gene)
    received = attn[:, focus_idx]
    top_k = 15
    top_src_idx = np.argsort(received)[::-1][:top_k]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh([genes[i] for i in top_src_idx][::-1], received[top_src_idx][::-1])
    ax.set_xlabel("attention weight")
    ax.set_title(f"Genes attending most TO '{focus_gene}' (saliency top gene)")
    plt.tight_layout()
    plt.show()


## 다음 단계 아이디어

- 이 노트북은 fold 1 하나만 썼습니다. 5개 fold 전부에 대해 임베딩/saliency를 뽑아서
  얼마나 일관적인지(fold 간 분산) 비교해볼 수 있습니다.
- `data/ccle_processed/`를 로드해서 GDSC2가 아닌 **완전히 새로운 데이터**(CCLE)에
  대한 추론으로 확장할 수 있습니다 -- 단, 같은 909개 유전자 순서로 정렬해야 합니다.
- `omicsdrp/scripts/InferenceModels/` 안의 다른 11개 조건과 비교해서, drug encoder나
  omics 조합에 따라 임베딩/attention 패턴이 어떻게 달라지는지 볼 수 있습니다.
- 재현 성능 숫자(nested-CV)가 필요하면 `omicsdrp/scripts/Results`를 참고하세요; 이 노트북의
  숫자는 참고용이지 리포트에 쓸 성능 지표가 아닙니다.
